# Basic Distributed Parallelism (Code-First)\n
\n
This notebook demonstrates core strategies with executable code: Data Parallelism, Model Split simulation, and Pipeline micro-batching simulation.

In [1]:
import math
import random
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Seed set to", SEED)

Seed set to 42


## Config

In [2]:
@dataclass(frozen=True)
class TrainConfig:
    n_samples: int = 20_000
    n_features: int = 32
    batch_size: int = 256
    learning_rate: float = 0.05
    epochs: int = 6
    workers: int = 2

cfg = TrainConfig()
cfg

TrainConfig(n_samples=20000, n_features=32, batch_size=256, learning_rate=0.05, epochs=6, workers=2)

## Data

In [3]:
def make_dataset(n_samples: int, n_features: int) -> Tuple[np.ndarray, np.ndarray]:
    x = np.random.randn(n_samples, n_features).astype(np.float64)
    true_w = np.random.randn(n_features, 1).astype(np.float64)
    logits = x @ true_w + 0.1 * np.random.randn(n_samples, 1)
    y = (logits > 0).astype(np.float64)
    return x, y

X, y = make_dataset(cfg.n_samples, cfg.n_features)
X.shape, y.shape

((20000, 32), (20000, 1))

## Model and Training Utilities

In [4]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))

def bce_loss(y_hat: np.ndarray, y_true: np.ndarray) -> float:
    eps = 1e-9
    y_hat = np.clip(y_hat, eps, 1 - eps)
    return float(-(y_true * np.log(y_hat) + (1 - y_true) * np.log(1 - y_hat)).mean())

def accuracy(y_hat: np.ndarray, y_true: np.ndarray) -> float:
    preds = (y_hat >= 0.5).astype(np.float64)
    return float((preds == y_true).mean())

def forward(xb: np.ndarray, w: np.ndarray, b: float) -> np.ndarray:
    return sigmoid(xb @ w + b)

def grad_step(xb: np.ndarray, yb: np.ndarray, w: np.ndarray, b: float) -> Tuple[np.ndarray, float, float]:
    y_hat = forward(xb, w, b)
    err = y_hat - yb
    grad_w = xb.T @ err / len(xb)
    grad_b = float(err.mean())
    loss = bce_loss(y_hat, yb)
    return grad_w, grad_b, loss

## Baseline: Single-Worker Training

In [5]:
def train_single_worker(x: np.ndarray, y: np.ndarray, cfg: TrainConfig) -> Dict[str, float]:
    w = np.zeros((cfg.n_features, 1), dtype=np.float64)
    b = 0.0
    start = time.perf_counter()

    for _ in range(cfg.epochs):
        idx = np.random.permutation(len(x))
        x_epoch = x[idx]
        y_epoch = y[idx]

        for i in range(0, len(x), cfg.batch_size):
            xb = x_epoch[i : i + cfg.batch_size]
            yb = y_epoch[i : i + cfg.batch_size]
            grad_w, grad_b, _ = grad_step(xb, yb, w, b)
            w -= cfg.learning_rate * grad_w
            b -= cfg.learning_rate * grad_b

    elapsed = time.perf_counter() - start
    y_hat = forward(x, w, b)
    return {
        "loss": bce_loss(y_hat, y),
        "accuracy": accuracy(y_hat, y),
        "seconds": elapsed,
    }

single_metrics = train_single_worker(X, y, cfg)
single_metrics

{'loss': 0.2028014375866223, 'accuracy': 0.99, 'seconds': 0.022386714000049324}

## Simulated Data Parallelism (Synchronous Gradient Averaging)

In [6]:
def train_data_parallel_sim(x: np.ndarray, y: np.ndarray, cfg: TrainConfig) -> Dict[str, float]:
    workers = cfg.workers
    w = np.zeros((cfg.n_features, 1), dtype=np.float64)
    b = 0.0
    micro_bs = cfg.batch_size // workers
    if micro_bs == 0:
        raise ValueError("batch_size must be >= workers")

    start = time.perf_counter()

    for _ in range(cfg.epochs):
        idx = np.random.permutation(len(x))
        x_epoch = x[idx]
        y_epoch = y[idx]

        for i in range(0, len(x), cfg.batch_size):
            xb = x_epoch[i : i + cfg.batch_size]
            yb = y_epoch[i : i + cfg.batch_size]
            if len(xb) < workers:
                continue

            grad_ws: List[np.ndarray] = []
            grad_bs: List[float] = []

            for w_id in range(workers):
                s = w_id * micro_bs
                e = len(xb) if w_id == workers - 1 else (w_id + 1) * micro_bs
                part_x = xb[s:e]
                part_y = yb[s:e]
                if len(part_x) == 0:
                    continue
                g_w, g_b, _ = grad_step(part_x, part_y, w, b)
                grad_ws.append(g_w)
                grad_bs.append(g_b)

            mean_gw = np.mean(np.stack(grad_ws, axis=0), axis=0)
            mean_gb = float(np.mean(np.array(grad_bs)))
            w -= cfg.learning_rate * mean_gw
            b -= cfg.learning_rate * mean_gb

    elapsed = time.perf_counter() - start
    y_hat = forward(x, w, b)
    return {
        "loss": bce_loss(y_hat, y),
        "accuracy": accuracy(y_hat, y),
        "seconds": elapsed,
    }

dp_metrics = train_data_parallel_sim(X, y, cfg)
dp_metrics

{'loss': 0.20285998762567223,
 'accuracy': 0.9904,
 'seconds': 0.0368241810000427}

## Model Split and Pipeline Timing Simulation

In [7]:
def estimate_pipeline_time(stage_times: List[float], micro_batches: int) -> float:
    # Approximation: fill + steady + drain for a simple pipeline
    depth = len(stage_times)
    warmup = sum(stage_times)
    bottleneck = max(stage_times)
    return warmup + max(0, micro_batches - 1) * bottleneck

def estimate_serial_time(stage_times: List[float], micro_batches: int) -> float:
    return micro_batches * sum(stage_times)

stage_times = [1.3, 1.0, 1.6, 1.1]  # relative units
micro_batches = 16
serial_t = estimate_serial_time(stage_times, micro_batches)
pipe_t = estimate_pipeline_time(stage_times, micro_batches)
speedup = serial_t / pipe_t
{"serial": serial_t, "pipeline": pipe_t, "speedup": round(speedup, 3)}

{'serial': 80.0, 'pipeline': 29.0, 'speedup': 2.759}

## Results

In [8]:
def pretty(metrics: Dict[str, float]) -> Dict[str, float]:
    return {k: round(v, 5) if isinstance(v, float) else v for k, v in metrics.items()}

print("Single worker:", pretty(single_metrics))
print("Data parallel simulation:", pretty(dp_metrics))
print("Pipeline timing speedup estimate:", round(speedup, 3))

Single worker: {'loss': 0.2028, 'accuracy': 0.99, 'seconds': 0.02239}
Data parallel simulation: {'loss': 0.20286, 'accuracy': 0.9904, 'seconds': 0.03682}
Pipeline timing speedup estimate: 2.759


## Framework Mapping (Starter Snippets)\n
\n
Use these in real multi-GPU setups:\n
- PyTorch DDP for data parallel.\n
- DeepSpeed ZeRO for memory sharding.\n
- Megatron-LM for tensor/pipeline parallel patterns.

In [9]:
ddp_snippet = '''
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

dist.init_process_group(backend="nccl")
model = MyModel().cuda()
model = DDP(model, device_ids=[local_rank])
'''
print(ddp_snippet)


import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

dist.init_process_group(backend="nccl")
model = MyModel().cuda()
model = DDP(model, device_ids=[local_rank])



## Summary\n
\n
- Data parallelism is usually the first strategy to implement.\n
- Model and pipeline splits are useful when model size or depth becomes limiting.\n
- Communication-aware planning is essential even at the basic level.